# Iceberg JDBC catalog (host)

Prerequisites: stack is up (`docker compose up -d` from `iceberg-jdbc`), ports **5432** (catalog DB), **9000** (MinIO), and the app has written at least one snapshot.

Install deps: `pip install -r ../../common/requirements-python.txt`

Debezium maps Postgres `public.demo_orders` to Iceberg namespace **`postgres_public`** and table **`demo_orders`**.

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "examples",
    type="jdbc",
    uri="jdbc:postgresql://127.0.0.1:5432/iceberg_catalog",
    **{
        "jdbc.user": "postgres",
        "jdbc.password": "postgres",
        "jdbc.schema-version": "V1",
        "warehouse": "s3://warehouse/iceberg-jdbc",
        "io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
        "s3.endpoint": "http://127.0.0.1:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
        "s3.path-style-access": "true",
    },
)

list(catalog.list_namespaces())

In [ ]:
tid = ("postgres_public", "demo_orders")
if catalog.table_exists(tid):
    table = catalog.load_table(tid)
    print(table.scan().to_arrow())
else:
    print("Table not present yet — run scripts/load.sh while the app is up.")